# 🤖 Chatbot RAG para Conceptos de Machine Learning

Este notebook implementa un sistema RAG (Retrieval-Augmented Generation) local, sin necesidad de GPU ni conexión a internet. Ideal para responder preguntas sobre conceptos básicos de machine learning.

- Modelo de embeddings: `all-MiniLM-L6-v2`
- Base vectorial: FAISS (CPU)
- Modelo de lenguaje: `google/flan-t5-base` (ejecutado en CPU)
- Framework: LangChain

### Instalación de dependencias
Esta celda instala las librerías necesarias para ejecutar el chatbot y procesar embeddings. Ejecuta solo la primera vez o cuando añades/actualizas dependencias.

In [4]:
# Instalación de dependencias (ejecutar solo la primera vez)
# Instala las librerías necesarias para ejecutar el chatbot y generar embeddings
%pip install -q langchain langchain-community sentence-transformers faiss-cpu torch transformers accelerate --no-warn-script-location
# Alternativa (descomentar si es necesario): #%pip install -q langchain langchain-community sentence-transformers faiss-cpu torch transformers accelerate
print("Dependencias instaladas correctamente.")

Note: you may need to restart the kernel to use updated packages.
Dependencias instaladas correctamente.


### ⚠️ Reiniciar el kernel
**Importante**: Después de instalar las dependencias por primera vez, necesitas **reiniciar el kernel** para que los paquetes se carguen correctamente.

En VS Code:
1. Click en el botón "Restart" en la barra superior del notebook, o
2. Usa el Command Palette (`Ctrl+Shift+P`) → "Notebook: Restart Kernel"

Después de reiniciar, ejecuta la celda de diagnóstico (siguiente celda) para verificar que `TextLoader` se importa correctamente.

### Diagnóstico del kernel y disponibilidad de LangChain
Esta celda imprime el intérprete Python usado por el kernel del notebook y prueba importar `langchain` y `TextLoader` desde `langchain.document_loaders`.
Ejecuta esta celda para comprobar si el kernel coincide con el intérprete donde instalaste paquetes (por ejemplo, la salida de `pip show langchain`).

In [4]:
# Diagnóstico: comprobar el intérprete del kernel y los imports relevantes
import sys, os
print("sys.executable:", sys.executable)
print("Python version:", sys.version.replace('\n', ' '))
print("sys.path (primeros 10):", sys.path[:10])

# Intentar importar langchain y el loader que usa el notebook
try:
    import langchain
    print("langchain importado. Versión:", getattr(langchain, '__version__', 'desconocida'))
except Exception as e:
    print("No se pudo importar langchain:", e)

try:
    from langchain.document_loaders import TextLoader
    print("TextLoader importado correctamente desde langchain.document_loaders.")
except Exception as e:
    print("No se pudo importar TextLoader desde langchain.document_loaders:", e)

# También mostrar ubicación del paquete si está disponible
try:
    import importlib.util, pkgutil
    loader = pkgutil.find_loader('langchain')
    print("pkgutil.find_loader('langchain'):", loader)
    spec = importlib.util.find_spec('langchain')
    print("importlib.util.find_spec('langchain'):", spec)
    if spec and getattr(spec, 'origin', None):
        print("langchain location:", spec.origin)
except Exception:
    pass

print("\nSi las importaciones fallan pero 'pip show langchain' muestra que está instalado en otro intérprete, selecciona el kernel correcto o ejecuta el comando de instalación dentro del notebook:")
print("%pip install --upgrade langchain langchain-community sentence-transformers faiss-cpu")

sys.executable: c:\ProgramData\anaconda3\python.exe
Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
sys.path (primeros 10): ['c:\\ProgramData\\anaconda3\\python313.zip', 'c:\\ProgramData\\anaconda3\\DLLs', 'c:\\ProgramData\\anaconda3\\Lib', 'c:\\ProgramData\\anaconda3', '', 'C:\\Users\\Jaf\\AppData\\Roaming\\Python\\Python313\\site-packages', 'C:\\Users\\Jaf\\AppData\\Roaming\\Python\\Python313\\site-packages\\win32', 'C:\\Users\\Jaf\\AppData\\Roaming\\Python\\Python313\\site-packages\\win32\\lib', 'C:\\Users\\Jaf\\AppData\\Roaming\\Python\\Python313\\site-packages\\Pythonwin', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages']
langchain importado. Versión: 1.0.3
No se pudo importar TextLoader desde langchain.document_loaders: No module named 'langchain.document_loaders'
pkgutil.find_loader('langchain'): <_frozen_importlib_external.SourceFileLoader object at 0x000001BDE6034410>
importlib.util.find_spec('langchain'): ModuleS

C:\Users\Jaf\AppData\Local\Temp\ipykernel_3676\4134048418.py:23: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  loader = pkgutil.find_loader('langchain')


### Instalación directa de langchain-community
Si el import de `TextLoader` falla después del reinicio, ejecuta esta celda para instalar `langchain-community` en el entorno correcto del kernel.

In [3]:
import sys
import subprocess

# Instalar langchain-community usando el ejecutable Python del kernel actual
print("Instalando langchain-community en:", sys.executable)
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "langchain-community", "sentence-transformers", "faiss-cpu"],
    capture_output=True,
    text=True
)
print(result.stdout)
if result.stderr:
    print("Errores/Advertencias:", result.stderr)
print("\n✅ Instalación completada. Ahora vuelve a ejecutar la celda de diagnóstico anterior.")

Instalando langchain-community en: c:\ProgramData\anaconda3\python.exe
Defaulting to user installation because normal site-packages is not writeable


✅ Instalación completada. Ahora vuelve a ejecutar la celda de diagnóstico anterior.


In [5]:
# Verificar instalación de langchain-community y probar la importación correcta
import sys

print("Verificando instalación de langchain-community...")
try:
    import langchain_community
    print("✅ langchain_community está instalado. Versión:", getattr(langchain_community, '__version__', 'desconocida'))
except ImportError as e:
    print("❌ langchain_community NO está instalado:", e)

print("\n--- Probando la importación correcta de TextLoader ---")
try:
    # En langchain >= 1.0, la importación correcta es desde langchain_community
    from langchain_community.document_loaders import TextLoader
    print("✅ TextLoader importado correctamente desde langchain_community.document_loaders")
except ImportError as e:
    print("❌ No se pudo importar TextLoader desde langchain_community.document_loaders:", e)

print("\nSi langchain_community no está instalado, ejecuta la celda anterior de instalación.")

Verificando instalación de langchain-community...
✅ langchain_community está instalado. Versión: 0.4.1

--- Probando la importación correcta de TextLoader ---
✅ TextLoader importado correctamente desde langchain_community.document_loaders

Si langchain_community no está instalado, ejecuta la celda anterior de instalación.


In [15]:
# Instalar todos los paquetes necesarios de langchain en sus versiones actualizadas
import sys
import subprocess

print("Instalando paquetes langchain completos...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", 
     "langchain>=0.3", "langchain-core", "langchain-community", "langchain-text-splitters"],
    capture_output=True,
    text=True
)
print(result.stdout[-800:] if len(result.stdout) > 800 else result.stdout)
if result.stderr:
    print("\nAdvertencias:", result.stderr[-300:] if len(result.stderr) > 300 else result.stderr)
print("\n✅ Paquetes langchain instalados/actualizados.")

Instalando paquetes langchain completos...
om requests>=2.0.0->langsmith<1.0.0,>=0.3.45->langchain-core) (3.4.4)


✅ Paquetes langchain instalados/actualizados.


### ⚠️ **IMPORTANTE: Reinicia el kernel AHORA**

Acabas de instalar/actualizar paquetes langchain. **Debes reiniciar el kernel** para que los cambios tengan efecto:

1. Click en "🔄 Restart" en la barra superior, O
2. `Ctrl+Shift+P` → "Notebook: Restart Kernel"

Después de reiniciar, **ejecuta las celdas en orden desde la celda 7 (crear corpus)** hasta la final.

In [6]:
# Verificar módulos de langchain disponibles después del reinicio
import sys
print("sys.executable:", sys.executable)

# Verificar paquetes langchain instalados
packages_to_check = ['langchain', 'langchain_core', 'langchain_community', 'langchain_text_splitters']
for pkg in packages_to_check:
    try:
        mod = __import__(pkg)
        version = getattr(mod, '__version__', 'desconocida')
        print(f"✅ {pkg}: versión {version}")
    except ImportError as e:
        print(f"❌ {pkg}: NO instalado - {e}")

# Probar imports críticos para RetrievalQA
print("\n--- Probando imports para RetrievalQA ---")
try:
    from langchain.chains.retrieval_qa.base import RetrievalQA
    print("✅ RetrievalQA importado desde langchain.chains.retrieval_qa.base")
except ImportError as e:
    print(f"❌ langchain.chains.retrieval_qa.base falló: {e}")
    
try:
    from langchain_core.runnables import RunnablePassthrough
    print("✅ RunnablePassthrough disponible (alternativa para construir cadena manualmente)")
except ImportError as e:
    print(f"❌ RunnablePassthrough no disponible: {e}")

sys.executable: c:\ProgramData\anaconda3\python.exe
✅ langchain: versión 1.0.3
✅ langchain_core: versión 1.0.3
✅ langchain_community: versión 0.4.1
✅ langchain_text_splitters: versión desconocida

--- Probando imports para RetrievalQA ---
❌ langchain.chains.retrieval_qa.base falló: No module named 'langchain.chains'
✅ RunnablePassthrough disponible (alternativa para construir cadena manualmente)


### Crear corpus de conceptos (documento de conceptos) de ML
Esta celda crea un texto con definiciones básicas de conceptos de Machine Learning y lo guarda en `ml_concepts_corpus.txt` para usarlo como corpus de ejemplo (podemos ampliarlo si queremos).

In [1]:
# Creamos un corpus básico de conceptos de ML
# Definimos un texto multilínea con definiciones breves de varios conceptos clave
ml_concepts_text = """
**Aprendizaje supervisado**: Tipo de aprendizaje automático donde el modelo se entrena con pares de entrada y salida etiquetados. Ejemplo: predecir el precio de una casa a partir de su tamaño.

**Aprendizaje no supervisado**: El modelo encuentra patrones en datos sin etiquetas. Ejemplo: agrupar clientes por comportamiento de compra (clustering).

**Regresión**: Tarea de predicción de un valor numérico continuo. Ejemplo: predecir la temperatura.

**Clasificación**: Tarea de asignar una etiqueta categórica. Ejemplo: detectar si un correo es spam o no.

**Overfitting**: Ocurre cuando un modelo aprende demasiado bien los datos de entrenamiento, incluyendo ruido, y falla en datos nuevos.

**Underfitting**: El modelo es demasiado simple y no captura patrones relevantes en los datos.

**Validación cruzada**: Técnica para evaluar la capacidad de generalización de un modelo dividiendo los datos en k particiones.

**Precisión (accuracy)**: Proporción de predicciones correctas sobre el total. No es ideal para datos desbalanceados.

**Random Forest**: Ensemble de árboles de decisión entrenados con subconjuntos aleatorios de datos y características.

**Gradiente descendente**: Algoritmo de optimización que ajusta los parámetros del modelo minimizando la pérdida.

**Red neuronal**: Modelo inspirado en el cerebro humano, compuesto por capas de neuronas artificiales. En scikit-learn, se implementa con MLPClassifier o MLPRegressor.
"""
# Guardamos el corpus en un archivo de texto
# Abrir (o crear) el archivo y escribir el corpus en UTF-8
with open("ml_concepts_corpus.txt", "w", encoding="utf-8") as f:
    f.write(ml_concepts_text)

print("✅ Corpus de conceptos de ML creado.")

✅ Corpus de conceptos de ML creado.


### Cargar y dividir documentos
Esta celda carga el corpus desde el archivo y lo segmenta en fragmentos (chunks) con solapamiento para facilitar la creación de embeddings y la recuperación semántica.

In [2]:
# Cargar y dividir documentos
# Importar un loader de texto y un divisor de texto que respeta estructuras
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Cargar el documento desde el archivo creado anteriormente
loader = TextLoader("ml_concepts_corpus.txt", encoding="utf-8")
documents = loader.load()

# Configurar el divisor: tamaño de chunk y solapamiento para mejor contexto
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = text_splitter.split_documents(documents)

# Mostrar cuántos fragmentos se crearon y ejemplos de los primeros dos
print(f"📚 Dividido en {len(chunks)} fragmentos.")
for i, chunk in enumerate(chunks[:2]):
    print(f"\nFragmento {i+1}: {chunk.page_content[:200]}...")

📚 Dividido en 6 fragmentos.

Fragmento 1: **Aprendizaje supervisado**: Tipo de aprendizaje automático donde el modelo se entrena con pares de entrada y salida etiquetados. Ejemplo: predecir el precio de una casa a partir de su tamaño....

Fragmento 2: **Aprendizaje no supervisado**: El modelo encuentra patrones en datos sin etiquetas. Ejemplo: agrupar clientes por comportamiento de compra (clustering).

**Regresión**: Tarea de predicción de un valo...


### Crear embeddings y base vectorial
Esta celda crea embeddings con un modelo ligero (MiniLM) y construye una base vectorial (FAISS) para realizar búsquedas semánticas.

In [3]:
# Crear embeddings y base vectorial
# Importar la clase de embeddings y FAISS para la base vectorial
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Configurar el modelo de embeddings (MiniLM) en CPU
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
 )

# Crear la base vectorial a partir de los fragmentos y el objeto de embeddings
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("🧠 Base vectorial creada con embeddings de MiniLM.")

C:\Users\Jaf\AppData\Local\Temp\ipykernel_17840\3204735115.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


🧠 Base vectorial creada con embeddings de MiniLM.


### Cargar modelo de lenguaje (Flan-T5)
Esta celda carga un modelo de LLM (Flan-T5 base) y lo prepara para generar respuestas; en CPU puede tardar y consumir memoria.

In [4]:
# Cargar modelo de lenguaje (Flan-T5 base)
# Importar torch y la utilidad pipeline de transformers
import torch
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

# Informar al usuario que la carga puede tardar en CPU
print("⏳ Cargando Flan-T5 (puede tardar 30-60 segundos en CPU)...")

# Crear el pipeline de generación de texto con el modelo Flan-T5
flan_pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    tokenizer="google/flan-t5-base",
    max_length=256,
    temperature=0.1,
    device=0 if torch.cuda.is_available() else -1
)

# Envolver el pipeline en un LLM compatible con LangChain
llm = HuggingFacePipeline(pipeline=flan_pipe)
print("🤖 Flan-T5 listo.")

⏳ Cargando Flan-T5 (puede tardar 30-60 segundos en CPU)...


Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🤖 Flan-T5 listo.


C:\Users\Jaf\AppData\Local\Temp\ipykernel_17840\2855672459.py:21: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=flan_pipe)


### Crear cadena RAG y función de consulta
Esta celda construye la cadena RetrievalQA (RAG) usando el LLM y el retriever, define la función `ask_ml_question` que realiza la consulta y muestra la respuesta junto con las fuentes.

In [7]:
# Crear cadena RAG y función de consulta
# En LangChain 1.0+, usamos RunnablePassthrough y formato de cadena moderno
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Crear el prompt template para la cadena RAG
template = """Responde la pregunta basándote SOLO en el siguiente contexto:

{context}

Pregunta: {question}

Respuesta:"""

prompt = ChatPromptTemplate.from_template(template)

# Función para formatear los documentos recuperados
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Construir la cadena RAG usando el patrón moderno de LangChain 1.0
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Función utilitaria para realizar una pregunta y mostrar la respuesta y las fuentes
def ask_ml_question(question):
    # Obtener documentos relevantes
    relevant_docs = retriever.invoke(question)
    
    # Ejecutar la cadena para obtener la respuesta
    answer = rag_chain.invoke(question)
    
    # Mostrar la pregunta y la respuesta formateada
    print(f"❓ Pregunta: {question}")
    print(f"💡 Respuesta: {answer}\n")
    
    # Mostrar las fuentes devueltas
    print("📄 Fuentes relevantes:")
    for i, doc in enumerate(relevant_docs, 1):
        # Mostrar un extracto corto de cada documento fuente
        print(f"  [{i}] {doc.page_content[:200]}...")

# Ejemplo de uso: hacer una pregunta sobre overfitting
ask_ml_question("¿Qué es el overfitting?")

❓ Pregunta: ¿Qué es el overfitting?
💡 Respuesta: Ocurre cuando un modelo aprende demasiado bien los datos de entrenamiento, incluyendo ruido, y falla en datos nuevos.

📄 Fuentes relevantes:
  [1] **Clasificación**: Tarea de asignar una etiqueta categórica. Ejemplo: detectar si un correo es spam o no.

**Overfitting**: Ocurre cuando un modelo aprende demasiado bien los datos de entrenamiento, i...
  [2] **Underfitting**: El modelo es demasiado simple y no captura patrones relevantes en los datos.

**Validación cruzada**: Técnica para evaluar la capacidad de generalización de un modelo dividiendo los ...


### Interfaz interactiva (opcional)
Esta celda crea un pequeño widget de entrada de texto para hacer preguntas interactivas al chatbot directamente desde el notebook.

In [ ]:
# Interfaz interactiva (opcional)
# Importar herramientas de IPython para widgets y display
from IPython.display import display
import ipywidgets as widgets

# Crear un widget de texto para que el usuario escriba su pregunta
text_input = widgets.Text(
    placeholder='Escribe tu pregunta sobre machine learning...',
    description='Pregunta:',
    )

# Crear un área de salida donde mostraremos la respuesta
output = widgets.Output()

# Función que se ejecuta al enviar el texto
def on_submit(sender):
    with output:
        # Limpiar la salida antes de imprimir la nueva respuesta
        output.clear_output()
        # Si el usuario ha escrito algo, realizar la consulta
        if sender.value.strip():
            ask_ml_question(sender.value.strip())
        else:
            print("Por favor, escribe una pregunta.")

# Conectar el evento 'submit' del widget con la función y mostrar los widgets
text_input.on_submit(on_submit)
display(text_input, output)

C:\Users\Jaf\AppData\Local\Temp\ipykernel_17840\1783444046.py:27: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_input.on_submit(on_submit)


Text(value='', description='Pregunta:', placeholder='Escribe tu pregunta sobre machine learning...')

Output()